# AR Cloud Multi-Agent RAG System

This notebook demonstrates a multi-agent RAG system with department routing, ChromaDB, OpenAI, and Langfuse.

## 1. Setup and Imports

In [ ]:
from src.config import CHROMA_PERSIST_DIRECTORY
from src.retrieval.chunking import DEPARTMENT_FOLDERS
from src.retrieval.retriever import retrieve_documents
from src.router import route_question
from src.agents.department_agent import answer_department_question
from src.orchestrator import process_question

print('Chroma directory:', CHROMA_PERSIST_DIRECTORY)

## 2. Document Collections and Vector Stores

Each department has an independent ChromaDB collection. Run the indexing command before executing retrieval examples.

In [ ]:
for department in DEPARTMENT_FOLDERS:
    print(department, f'{department}_documents')

## 3. Specialized Department Agents

In [ ]:
question = 'How do I request annual leave?'
documents = retrieve_documents(question, 'hr', k=3)
print('Retrieved chunks:', len(documents))
print('Document IDs:', [doc.metadata['document_id'] for doc in documents])

### Department Agent Examples

The system uses three specialized RAG agents: Human Resources, Technology, and Finance. Each agent retrieves information from its own ChromaDB collection and generates answers grounded in department-specific documents.

The following examples demonstrate the behavior of each specialized agent.

In [ ]:

department_questions = {
    "hr": "How do I request annual leave?",
    "technology": "How do I report a technical incident?",
    "finance": "How do I submit a reimbursement request?",
}

for department, question in department_questions.items():
    result = answer_department_question(
        question=question,
        department=department,
        k=3,
    )

    print(f"\nDepartment: {department.upper()}")
    print(f"Question: {question}")
    print(f"Answer: {result['answer']}")
    print("Sources:")
    for source in result["sources"]:
        print(f"  - {source['document_id']}: {source['source']}")
    print("-" * 60)


## 4. Router and Orchestrator

In [ ]:
routing = route_question(question)
print('Department:', routing.department)
print('Reason:', routing.reason)

In [ ]:
result = process_question(question)
print('Answer:', result['answer'])
print('Sources:', result['sources'])

## 5. Tests and Examples

In [ ]:
from pathlib import Path
from src.evaluation import load_test_queries

queries = load_test_queries(Path('test_queries.json'))
print('Available evaluation queries:', len(queries))
for query in queries[:3]:
    print(query['id'], query['question'])

### Evaluation Results

The evaluation dataset contains 15 queries covering the three departments and edge cases.

The evaluation measures routing accuracy, retrieval hit rate, and answer quality using an LLM judge. The quality score ranges from 0 to 10.

The following cell loads the previously generated evaluation report without making additional OpenAI API calls.

In [ ]:

import json
from pathlib import Path

report_path = Path("evaluation/report.json")

with report_path.open(encoding="utf-8") as file:
    report = json.load(file)

metrics = report["metrics"]

print("Evaluation Summary")
print("-" * 40)
print(f"Total queries: {metrics['total_queries']}")
print(f"Routing accuracy: {metrics['routing_accuracy']:.2%}")
print(f"Retrieval hit rate: {metrics['retrieval_hit_rate']:.2%}")
print(
    "Average answer quality: "
    f"{metrics['average_answer_quality']:.2f}/10"
)
print(f"Quality evaluations: {metrics['quality_evaluated']}")


In [ ]:

print(f"{'Query':<12} {'Routing':<12} {'Retrieval':<12} {'Quality'}")
print("-" * 50)

for result in report["results"]:
    routing = result["routing_correct"]
    retrieval = result["retrieval_hit"]
    quality = result.get("answer_quality_score")

    routing_text = "N/A" if routing is None else str(routing)
    retrieval_text = "N/A" if retrieval is None else str(retrieval)
    quality_text = "N/A" if quality is None else f"{quality}/10"

    print(
        f"{result['id']:<12} "
        f"{routing_text:<12} "
        f"{retrieval_text:<12} "
        f"{quality_text}"
    )


## 6. Langfuse Observability

The orchestrator records routing, retrieval, and generation traces in Langfuse.

Run the orchestration cells above and inspect the resulting traces in the configured Langfuse project.

### Trace Inspection and LLM Judge Scores

Langfuse records the execution of the multi-agent RAG workflow, including:

- Department routing and LLM generation.
- Document retrieval from the department-specific vector store.
- Input questions, generated answers, and execution metadata.
- Model usage, latency, and estimated costs.
- LLM judge scores associated with the corresponding traces.

The evaluation pipeline uses the Langfuse Score API to record answer quality scores from 0 to 10.

To inspect the results, open the configured Langfuse project and navigate to **Tracing** for execution details or **Scores** for answer quality evaluations.